# dllm on Colab

Masked diffusion LM (ModernBERT) 프리트레인 -> 샘플링 -> SFT -> 인퍼런스.

로컬 RTX 4060 Ti(8GB) 실측이 **4,300 tok/s** 였습니다. Colab GPU별 대략치:

| GPU | VRAM | precision | 예상 tok/s |
|---|---|---|---|
| T4 (무료) | 16GB | fp16 (bf16 미지원) | ~6k |
| L4 | 24GB | bf16 | ~15k |
| A100 40GB | 40GB | bf16 | ~50k |

**런타임 유형을 GPU로 바꾸고 시작하세요.** 아래 셀을 위에서부터 순서대로 실행하면 됩니다.

## 1. GPU 확인

In [ ]:
!nvidia-smi

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())

## 2. 의존성 설치

ModernBERT는 transformers 4.48 이상이 필요합니다. 설치 후 **런타임 재시작 요구가 뜨면 재시작**하고
이 셀 아래부터 다시 실행하세요.

In [ ]:
!pip install -q -U "transformers>=4.48" datasets accelerate safetensors tokenizers wandb

## 3. 코드 가져오기 (git clone)

코드를 고칠 때마다 로컬에서 push 하고 여기서 이 셀만 다시 실행하면 됩니다.

리포가 비공개라면 URL에 토큰을 끼워 넣어야 하는데, 노트북에 토큰이 그대로 남지 않게 이렇게 받으세요:

```python
from getpass import getpass
token = getpass("GitHub token: ")
REPO_URL = f"https://{token}@github.com/Jaehyeon-kr/DLLM.git"
```

In [ ]:
REPO_URL = "https://github.com/Jaehyeon-kr/DLLM.git"

import os, shutil

CODE_DIR = "/content/dllm"

### 이미 받아둔 게 있으면 지우고 새로 받는다. 매 세션 최신을 쓰는 게 헷갈릴 일이 없다 ###
if os.path.exists(CODE_DIR):
    shutil.rmtree(CODE_DIR)

!git clone --depth 1 {REPO_URL} {CODE_DIR}

print(sorted(f for f in os.listdir(CODE_DIR) if f.endswith(".py")))

### Google Drive 마운트 (체크포인트 보관용)

코드는 git에서 받더라도 체크포인트는 Drive에 두는 게 좋습니다. Colab 세션이 끊기면 `/content`는
전부 날아갑니다. Drive를 안 쓸 거면 이 셀은 건너뛰고 다음 셀의 `WORK_DIR`을 `/content/dllm_exp`로
바꾸세요.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 4. 설정

GPU를 보고 precision과 배치를 자동으로 잡습니다.

`per_gpu_batch_size`가 **실효 배치**이고 `gradient_accumulation_steps`로 나눈 값이 실제 미니배치입니다
(`pretrain.py`의 `mini_batchsize = per_gpu_batch_size // gradient_accumulation_steps`).
아래는 실효 배치를 128로 고정하고, VRAM이 허락하는 만큼 미니배치를 키워 accum을 줄이는 방식입니다.

In [ ]:
import os, torch, math

### 체크포인트는 Drive에 두는 게 안전합니다. 세션이 끊겨도 남습니다 ###
WORK_DIR   = "/content/drive/MyDrive/dllm_exp"   # Drive를 안 쓴다면 "/content/dllm_exp"
DATA_DIR   = "/content/prepped_data"             # 재생성이 싸니 로컬 디스크로 충분
SFT_DATA_DIR = "/content/prepped_sft_data"

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1024**3

### T4는 Turing이라 bf16이 없습니다. 이 경우 fp16으로 떨어뜨립니다 ###
MIXED_PRECISION = "bf16" if torch.cuda.is_bf16_supported() else "fp16"

### seq 1024 기준으로 대략 이 정도가 안전선입니다 ###
if   vram_gb >= 38: MINI_BATCH = 16
elif vram_gb >= 22: MINI_BATCH = 8
elif vram_gb >= 15: MINI_BATCH = 4
else:               MINI_BATCH = 2

EFFECTIVE_BATCH = 128
ACCUM = max(1, EFFECTIVE_BATCH // MINI_BATCH)

### C4 샤드 하나가 약 170M 토큰 / 디스크 330MB 입니다 ###
NUM_C4_SHARDS = 4

print(f"gpu             : {gpu_name} ({vram_gb:.1f} GB)")
print(f"mixed_precision : {MIXED_PRECISION}")
print(f"mini batch      : {MINI_BATCH}")
print(f"effective batch : {EFFECTIVE_BATCH}  (accum {ACCUM})")
print(f"tokens / step   : {EFFECTIVE_BATCH * 1024 / 1000:.0f}k")

os.environ["PYTHONPATH"] = CODE_DIR
os.chdir(CODE_DIR)

## 5. 프리트레인 데이터 준비

C4를 받아서 토크나이즈하고 1024 길이로 패킹합니다. 샤드 4개면 15~25분 정도 걸립니다.

In [ ]:
!python prepare_data.py \
    --test_split_pct 0.005 \
    --context_length 1024 \
    --path_to_data_store {DATA_DIR} \
    --dataset_split_seed 42 \
    --num_workers 4 \
    --hf_model_name "answerdotai/ModernBERT-base" \
    --num_c4_shards {NUM_C4_SHARDS}

## 6. 스텝 수 정하기

먼저 10스텝만 돌려서 이 GPU의 실제 s/step을 재고, 주어진 시간 예산에 맞는 `num_training_steps`를
역산합니다. Colab 세션은 보통 12시간에서 끊기니 예산은 그보다 짧게 잡으세요.

In [ ]:
import time, subprocess

PROBE_STEPS = 10
start = time.time()
subprocess.run([
    "python", "pretrain.py",
    "--experiment_name", "probe",
    "--working_directory", "/content/probe",
    "--hf_model_name", "answerdotai/ModernBERT-base",
    "--mixed_precision", MIXED_PRECISION,
    "--path_to_prepped_data", DATA_DIR,
    "--num_workers", "2",
    "--per_gpu_batch_size", str(EFFECTIVE_BATCH),
    "--gradient_accumulation_steps", str(ACCUM),
    "--num_training_steps", str(PROBE_STEPS),
    "--num_warmup_steps", "1",
    "--evaluation_interval", "10000",
    "--checkpoint_interval", "10000",
    "--logging_steps", "10000",
    "--max_grad_norm", "1.0",
    "--learning_rate", "1e-4",
    "--no-log_wandb",
], check=True)
elapsed = time.time() - start

### 모델 로딩/컴파일 오버헤드가 섞여 있으니 넉넉히 30초를 빼줍니다 ###
sec_per_step = max(0.1, (elapsed - 30) / PROBE_STEPS)
tok_per_sec  = EFFECTIVE_BATCH * 1024 / sec_per_step

BUDGET_HOURS = 8
NUM_TRAINING_STEPS = int(BUDGET_HOURS * 3600 / sec_per_step)

print(f"\n{sec_per_step:.2f} s/step  |  {tok_per_sec:,.0f} tok/s")
print(f"{BUDGET_HOURS}시간 예산 -> num_training_steps = {NUM_TRAINING_STEPS:,}")
print(f"총 학습 토큰 {NUM_TRAINING_STEPS * EFFECTIVE_BATCH * 1024 / 1e9:.2f}B "
      f"(코퍼스 대비 {NUM_TRAINING_STEPS * EFFECTIVE_BATCH * 1024 / (NUM_C4_SHARDS * 170e6):.1f} 에폭)")

## 7. 프리트레인

- `max_grad_norm`은 반드시 **1.0**입니다. 원본 `pretrain.sh`에 있던 1e-4는 오타이고, 그 값이면 매 업데이트가
  1만분의 1로 줄어들어 학습이 사실상 멈춥니다.
- `num_warmup_steps`는 전체의 1~2% 정도가 무난합니다.
- `checkpoint_interval`을 너무 촘촘히 잡지 마세요. `accelerator.save_state`는 옵티마이저까지 저장해서
  한 번에 1.8GB 가까이 씁니다. Drive 용량이 순식간에 찹니다.
- `evaluation_interval`도 마찬가지입니다. eval 한 번이 수십 초에서 수 분입니다.

In [ ]:
WARMUP  = max(10, NUM_TRAINING_STEPS // 50)     # 2%
EVAL_IV = max(50, NUM_TRAINING_STEPS // 20)     # 20번
CKPT_IV = max(50, NUM_TRAINING_STEPS // 10)     # 10개
LOG_IV  = max(10, NUM_TRAINING_STEPS // 200)

print(f"warmup {WARMUP} | eval every {EVAL_IV} | ckpt every {CKPT_IV} | log every {LOG_IV}")

!python pretrain.py \
    --experiment_name "dllm" \
    --working_directory {WORK_DIR} \
    --hf_model_name "answerdotai/ModernBERT-base" \
    --mixed_precision {MIXED_PRECISION} \
    --path_to_prepped_data {DATA_DIR} \
    --num_workers 2 \
    --per_gpu_batch_size {EFFECTIVE_BATCH} \
    --gradient_accumulation_steps {ACCUM} \
    --num_training_steps {NUM_TRAINING_STEPS} \
    --max_grad_norm 1.0 \
    --lr_scheduler_type cosine \
    --num_warmup_steps {WARMUP} \
    --logging_steps {LOG_IV} \
    --evaluation_interval {EVAL_IV} \
    --checkpoint_interval {CKPT_IV} \
    --learning_rate 1e-4 \
    --weight_decay 2e-5 \
    --no-log_wandb

## 8. 샘플링

`--show_steps`를 주면 스텝마다 `_`(아직 안 채워진 위치)가 줄어드는 게 보입니다.
`--path_to_checkpoint`는 디렉토리를 받습니다 (`sample.py`는 그 안에서 model.safetensors를 찾습니다).

In [ ]:
import glob, os

ckpts = sorted(glob.glob(f"{WORK_DIR}/dllm/checkpoint_*"),
               key=lambda p: int(p.rsplit("_", 1)[1]))
CKPT = ckpts[-1]
print("using", CKPT)

!python sample.py \
    --hf_model_name "answerdotai/ModernBERT-base" \
    --path_to_checkpoint {CKPT} \
    --prompt "The capital of France" \
    --gen_length 128 \
    --num_steps 128 \
    --block_length 32 \
    --strategy confidence \
    --temperature 1.0 \
    --top_p 0.95 \
    --num_samples 2 \
    --seed 42 \
    --show_steps

## 9. SFT 데이터 준비 (Alpaca)

52k 샘플이라 1~2분이면 끝납니다.

In [ ]:
!python prepare_data_sft.py \
    --test_split_pct 0.01 \
    --context_length 1024 \
    --path_to_data_store {SFT_DATA_DIR} \
    --dataset_split_seed 42 \
    --num_workers 4 \
    --hf_model_name "answerdotai/ModernBERT-base"

## 10. SFT

프리트레인과 달리 데이터가 51,481개로 고정이라 에폭 기준으로 스텝을 잡습니다. 보통 2~3에폭입니다.
LR도 프리트레인보다 한 자릿수 낮춥니다.

`--path_to_pretrained_checkpoint`는 디렉토리가 아니라 **`.safetensors` 파일 경로**입니다
(`finetune_sft.py`가 `load_file`로 직접 읽습니다).

In [ ]:
SFT_EPOCHS = 3
SFT_TRAIN_SAMPLES = 51481
SFT_STEPS = SFT_EPOCHS * SFT_TRAIN_SAMPLES // EFFECTIVE_BATCH

PRETRAINED = f"{CKPT}/model.safetensors"

### ! 매직 안의 {} 치환은 단순한 이름일 때 가장 안전하니 미리 계산해 둡니다 ###
SFT_WARMUP  = max(10, SFT_STEPS // 50)
SFT_LOG_IV  = max(10, SFT_STEPS // 100)
SFT_EVAL_IV = max(50, SFT_STEPS // 10)
SFT_CKPT_IV = max(50, SFT_STEPS // 5)

print(f"{SFT_STEPS} steps from {PRETRAINED}")

!python finetune_sft.py \
    --experiment_name "dllm_sft" \
    --working_directory {WORK_DIR} \
    --path_to_pretrained_checkpoint {PRETRAINED} \
    --hf_model_name "answerdotai/ModernBERT-base" \
    --mixed_precision {MIXED_PRECISION} \
    --path_to_prepped_data {SFT_DATA_DIR} \
    --num_workers 2 \
    --per_gpu_batch_size {EFFECTIVE_BATCH} \
    --gradient_accumulation_steps {ACCUM} \
    --num_training_steps {SFT_STEPS} \
    --max_grad_norm 1.0 \
    --lr_scheduler_type cosine \
    --num_warmup_steps {SFT_WARMUP} \
    --logging_steps {SFT_LOG_IV} \
    --evaluation_interval {SFT_EVAL_IV} \
    --checkpoint_interval {SFT_CKPT_IV} \
    --learning_rate 1e-5 \
    --weight_decay 2e-5 \
    --no-log_wandb

## 11. 인퍼런스

SFT한 모델에 대화 템플릿을 씌워서 물어봅니다. `--strategy`는 `random` / `low_confidence` 두 개뿐이고,
`sample.py`의 `confidence`와는 이름만 다른 별개 구현입니다.

In [ ]:
!python inference.py \
    --safetensors_path {WORK_DIR}/dllm_sft/final_model/model.safetensors \
    --seq_len 256 \
    --num_steps 256 \
    --strategy low_confidence \
    --hf_model_name "answerdotai/ModernBERT-base" \
    --prompt "What is artificial intelligence?"

## 참고 — 세션이 끊겼을 때

`WORK_DIR`을 Drive로 잡아뒀다면 체크포인트가 남아 있습니다. 다만 `pretrain.py`에는 재개(resume) 경로가
없어서, 이어서 돌리려면 `finetune_sft.py`처럼 체크포인트를 초기 가중치로 읽는 방식으로 쓰거나
`accelerator.load_state`를 추가해야 합니다.